# Graph 기반 GraphRAG — Text2Cypher (`neo4j-graphrag`)

벡터 검색은 '의미가 비슷한 텍스트'를 찾지만, **관계형 질문**("A 와 함께 일한 사람은?")엔 약하다. 그래프의 진짜 힘은 **Cypher 쿼리로 관계를 탐색**하는 것.

**`Text2CypherRetriever`** 는 자연어를 Cypher 로 바꿔 그래프를 직접 조회한다. (04 노트북에서 이 과정을 LangGraph 로 직접 구현했다면, 여기서는 라이브러리가 제공하는 버전)

> Neo4j + `OPENAI_API_KEY` 필요. 예시는 POLE(Person-Object-Location-Event) 경찰 도메인 그래프.

## Neo4j 연결 & LLM
`.env` 에 `NEO4J_URI` / `NEO4J_USERNAME` / `NEO4J_PASSWORD` / `OPENAI_API_KEY` 를 넣는다. (README 의 'Neo4j 준비' 참고)

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
URI = os.environ["NEO4J_URI"]
AUTH = (os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Neo4j 연결 성공")

In [ ]:
from neo4j_graphrag.llm import OpenAILLM

llm = OpenAILLM(model_name="gpt-4o")

## 1. DB 스키마 준비
Text2Cypher 는 스키마를 알아야 정확한 쿼리를 만든다. Neo4j 내장 프로시저로 추출하거나 직접 기술한다.

In [ ]:
from collections import defaultdict

def get_schema() -> str:
    schema = ""
    with driver.session() as session:
        rel_types = session.run(
            "MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS f, type(r) AS t, labels(b) AS to"
        )
        rels = set()
        for r in rel_types:
            if r['f'] and r['to']:
                rels.add(f"(:{r['f'][0]})-[:{r['t']}]->(:{r['to'][0]})")
    schema += "Relationships:\n" + "\n".join(sorted(rels))
    return schema

neo4j_schema = get_schema()
print(neo4j_schema)

## 2. Text2CypherRetriever — 자연어 → Cypher 검색

few-shot 예시(질문↔Cypher 쌍)와 스키마를 주면, 질문을 Cypher 로 바꿔 실행한다.

In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever

examples = [
    "USER INPUT: Piccadilly 에서 발생한 범죄 건수는? "
    "QUERY: MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {address: 'Piccadilly'}) RETURN count(c)",
]

retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
)

result = retriever.search(query_text="현재 수사 중인 범죄 사건의 개수는?")
print("생성된 Cypher:", result.metadata["cypher"])
print("결과:", result.items)

## 3. GraphRAG 파이프라인
Text2Cypher 검색 결과를 근거로 답변 생성.

In [ ]:
from neo4j_graphrag.generation import GraphRAG

graph_rag = GraphRAG(retriever, llm)
response = graph_rag.search(
    query_text="범죄자는 아니지만, 범죄자를 많이 알고 있는 사람은?",
    return_context=True,
)
print(response.answer)

## 정리

- **Text2CypherRetriever**: 자연어 → Cypher → 그래프 조회 (관계형 질문에 강함)
- few-shot 예시 + 스키마가 쿼리 정확도의 핵심
- 벡터 검색(01) vs Cypher 검색(02): 의미 유사 vs 관계 탐색 — 상호 보완적

다음: 둘을 결합한 Vector + Graph 검색.